In [20]:
import os 
import re
from collections import Counter
from pathlib import Path

In [21]:
def get_folders(path):
    try:
        folders = os.listdir(path)
        
        folders = [folder for folder in folders if os.path.isdir(os.path.join(path, folder))]

        return folders

    except FileNotFoundError:
        print(f"Lỗi: Đường dẫn '{path}' không tồn tại.")
        return []
    except PermissionError:
        print(f"Lỗi: Không có quyền truy cập vào đường dẫn '{path}'.")
        return []

folders = get_folders("../data")
print(f"Số lượng folders: {len(folders)}")
# for item in folders:
#     print(item)


Số lượng folders: 47


In [22]:
def extract_scenario(folder_name):
    parts = folder_name.split('_', 2)
    if len(parts) < 3:
        return None
    
    remaining = parts[2]
    
    # Sử dụng Regex để kiểm tra xem phần còn lại có bắt đầu bằng "[Số thứ tự]_" hay không
    # Ví dụ: "1_engine_failure" sẽ khớp, còn "engine_failure" hoặc "no_failure" thì không
    match = re.match(r"^(\d+)_(.*)$", remaining)
    if match:
        return match.group(2)
    else:
        return remaining
    
all_scenarios = []
for item in folders:
    scenario = extract_scenario(item)
    if scenario:
        all_scenarios.append(scenario)

print("Thóng kê các loại lỗi:")
scenario_counts = Counter(all_scenarios)
for scenario, count in scenario_counts.items():
    print(f"{scenario}: {count} thư mục")


Thóng kê các loại lỗi:
no_ground_truth: 1 thư mục
engine_failure: 7 thư mục
engine_failure_with_emr_traj: 16 thư mục
no_failure: 10 thư mục
elevator_failure: 2 thư mục
left_aileron__right_aileron__failure: 1 thư mục
rudder_right_failure: 2 thư mục
rudder_left_failure: 1 thư mục
rudder_zero__left_aileron_failure: 1 thư mục
both_ailerons_failure: 1 thư mục
right_aileron_failure: 2 thư mục
left_aileron_failure: 2 thư mục
right_aileron_failure_with_emr_traj: 1 thư mục


# 📊 Bảng tóm tắt các kịch bản bay trong bộ dữ liệu ALFA
*(Tổng cộng: 47 thư mục chuyến bay)*

Dữ liệu được chia thành **4 nhóm chính** dựa trên tình trạng của máy bay và loại lỗi được kích hoạt.

---

### 🟢 1. Nhóm Dữ liệu Chuẩn (Khỏe mạnh)
*Dùng để làm dữ liệu nền (Baseline) hoặc gán nhãn `0` (Normal) khi huấn luyện AI.*

*   **`no_failure` (10 thư mục):** Máy bay hoạt động hoàn toàn bình thường từ lúc cất cánh đến khi hạ cánh. Mọi chỉ số thực tế (`_m`) đều bám sát lệnh điều khiển (`_c`).

---

### 🔴 2. Nhóm Lỗi Động Cơ (Engine Failures)
*Máy bay bị tắt động cơ đột ngột, làm mất hoàn toàn lực đẩy. Đặc trưng của lỗi này là vận tốc gió (Airspeed) giảm nhanh và rớt độ cao.*

*   **`engine_failure` (7 thư mục):** Lỗi động cơ tiêu chuẩn. Máy bay rơi vào trạng thái lướt tự do và từ từ rơi xuống.
*   **`engine_failure_with_emr_traj` (16 thư mục):** Lỗi động cơ nhưng hệ thống tự lái có kích hoạt **Quỹ đạo khẩn cấp (Emergency Trajectory)**. Tức là AI của máy bay tự động nhận diện được lỗi và điều khiển máy bay lượn xuống theo một đường bay an toàn được tính toán sẵn.

---

### 🟠 3. Nhóm Lỗi Kẹt Cánh Lái (Control Surface Failures)
*Động cơ vẫn hoạt động, nhưng các cánh tà dùng để bẻ lái bị kẹt cứng. Nhóm này ảnh hưởng trực tiếp đến 3 trục tư thế của máy bay (Roll, Pitch, Yaw).*

**A. Lỗi Cánh Liệng (Aileron - Chịu trách nhiệm góc Roll / Lật nghiêng)**
*   **`left_aileron_failure` (2 thư mục):** Kẹt cánh liệng bên trái.
*   **`right_aileron_failure` (2 thư mục):** Kẹt cánh liệng bên phải.
*   **`left_aileron__right_aileron__failure` (1 thư mục) & `both_ailerons_failure` (1 thư mục):** Kẹt cả hai cánh liệng cùng lúc. Máy bay mất hoàn toàn khả năng giữ thăng bằng ngang, dễ bị lộn vòng.
*   **`right_aileron_failure_with_emr_traj` (1 thư mục):** Kẹt cánh liệng phải và có kích hoạt hệ thống hạ cánh khẩn cấp.

**B. Lỗi Cánh Nâng/Hạ (Elevator - Chịu trách nhiệm góc Pitch / Ngóc, chúc mũi)**
*   **`elevator_failure` (2 thư mục):** Kẹt cánh đuôi ngang. Máy bay mất khả năng điều khiển mũi, dễ bị cắm thẳng xuống đất hoặc ngóc lên quá đà gây thất tốc (stall).

**C. Lỗi Cánh Hướng (Rudder - Chịu trách nhiệm góc Yaw / Rẽ trái, phải)**
*   **`rudder_left_failure` (1 thư mục):** Cánh đuôi dọc bị kẹt lệch sang trái, khiến mũi máy bay liên tục bị lỉa sang trái.
*   **`rudder_right_failure` (2 thư mục):** Cánh đuôi dọc bị kẹt lệch sang phải.

**D. Lỗi Kép / Hỗn hợp (Compound Failures)**
*   **`rudder_zero__left_aileron_failure` (1 thư mục):** Lỗi cực kỳ phức tạp. Cánh hướng (Rudder) bị kẹt ở vị trí `0` (vị trí trung tâm không bẻ lái được) ĐỒNG THỜI cánh liệng trái (`left_aileron`) cũng bị kẹt. 

---

### ⚪ 4. Nhóm Không sử dụng được
*   **`no_ground_truth` (1 thư mục):** Chuyến bay bị thiếu file ghi nhận thời điểm xảy ra lỗi. **Lưu ý:** Cần loại bỏ thư mục này ra khỏi quá trình huấn luyện mô hình (Supervised Learning) vì không có nhãn dữ liệu (Labels).

In [2]:
import os

def get_files(path):
    try:
        all_items = os.listdir(path)
        
        csv_files = [
            file for file in all_items 
            if os.path.isfile(os.path.join(path, file)) and file.lower().endswith('.csv')
        ]

        return csv_files

    except FileNotFoundError:
        print(f"Lỗi: Đường dẫn '{path}' không tồn tại.")
        return []
    except PermissionError:
        print(f"Lỗi: Không có quyền truy cập vào đường dẫn '{path}'.")
        return []

# Ví dụ sử dụng:
path = r"E:\Study\FPT\S4\DAP391m\project_main\data\carbonZ_2018-09-11-17-27-13_2_both_ailerons_failure"
csv_list = get_files(path)
print(f"Số lượng file CSV: {len(csv_list)}")
# for file in csv_list:
#     print(file)

Số lượng file CSV: 34


In [43]:
def extract_function(file_name, folder_name):
    prefix = f"{folder_name}-"
    
    if file_name.startswith(prefix):
        return file_name[len(prefix):-4]
    return file_name[:-4]

all_functions = []
for folder_name in folders:
    csv_list = get_files("../data/" + folder_name)
    for file in csv_list:
        func = extract_function(file, folder_name)
        if func:
            all_functions.append(func)

# Thống kê số lượng từng loại chức năng
print("\nThống kê các loại chức năng:")
function_counts = Counter(all_functions)
# print(len(function_counts))
for func, count in function_counts.items():
    print(f"{func}: {count} file")

    


Thống kê các loại chức năng:
diagnostics: 46 file
mavctrl-path_dev: 38 file
mavctrl-rpy: 40 file
mavlink-from: 45 file
mavros-battery: 44 file
mavros-global_position-compass_hdg: 28 file
mavros-global_position-global: 28 file
mavros-global_position-local: 28 file
mavros-global_position-raw-fix: 28 file
mavros-global_position-raw-gps_vel: 28 file
mavros-global_position-rel_alt: 28 file
mavros-imu-atm_pressure: 44 file
mavros-imu-data: 44 file
mavros-imu-data_raw: 44 file
mavros-imu-mag: 44 file
mavros-imu-temperature: 44 file
mavros-local_position-odom: 40 file
mavros-local_position-pose: 40 file
mavros-local_position-velocity: 28 file
mavros-nav_info-airspeed: 40 file
mavros-nav_info-errors: 44 file
mavros-nav_info-pitch: 44 file
mavros-nav_info-roll: 44 file
mavros-nav_info-velocity: 40 file
mavros-nav_info-yaw: 44 file
mavros-rc-in: 45 file
mavros-rc-out: 45 file
mavros-setpoint_raw-local: 34 file
mavros-setpoint_raw-target_global: 28 file
mavros-state: 45 file
mavros-time_reference

# 📂 Phân nhóm & Giải thích chức năng các file CSV trong ALFA Dataset
*(Tổng hợp từ 38 loại file chức năng khác nhau)*

Để thuận tiện cho việc xử lý dữ liệu và huấn luyện mô hình AI, các file chức năng được chia thành **5 nhóm chính** dựa trên bản chất vật lý và vai trò của chúng.

---

### 🔴 Nhóm 1: Nhãn Trạng Thái Lỗi (Failure Ground Truth)
*Đây là nhóm chứa nhãn (labels) mục tiêu của mô hình. Giá trị `1` (hoặc `true`) cho biết lỗi đang xảy ra.*

| Tên File | Số lượng | Giải thích chức năng | Mức độ ưu tiên |
| :--- | :---: | :--- | :--- |
| **`failure_status-engines`** | 23 file | Nhãn lỗi động cơ (tắt động cơ, mất lực đẩy). | **Bắt buộc dùng** |
| **`failure_status-aileron`** | 5 file | Nhãn lỗi kẹt cánh liệng (mất thăng bằng ngang). | **Bắt buộc dùng** |
| **`failure_status-rudder`** | 3 file | Nhãn lỗi kẹt cánh hướng ở đuôi dọc. | **Bắt buộc dùng** |
| **`failure_status-elevator`** | 2 file | Nhãn lỗi kẹt cánh nâng/hạ ở đuôi ngang. | **Bắt buộc dùng** |

---

### 🟡 Nhóm 2: Lệnh, Tư Thế & Sai Số Bám (Core Features)
*Nhóm đắt giá nhất để phát hiện lỗi. Khi hệ thống gặp sự cố vật lý, sai lệch giữa lệnh điều khiển (`commanded`) và giá trị đo được thực tế (`measured`) sẽ tăng vọt.*

| Tên File | Số lượng | Giải thích chức năng | Mức độ ưu tiên |
| :--- | :---: | :--- | :--- |
| **`mavros-nav_info-roll`** | 44 file | Góc nghiêng cánh trái/phải (gồm lệnh vs thực tế). | ⭐ **Rất cao** |
| **`mavros-nav_info-pitch`** | 44 file | Góc chúc/ngóc mũi máy bay (gồm lệnh vs thực tế). | ⭐ **Rất cao** |
| **`mavros-nav_info-yaw`** | 44 file | Góc hướng bay trái/phải (gồm lệnh vs thực tế). | ⭐ **Rất cao** |
| **`mavros-nav_info-airspeed`** | 40 file | Tốc độ gió tương đối (gồm lệnh thực tế). | ⭐ **Rất cao** |
| **`mavros-nav_info-velocity`** | 40 file | Vận tốc di chuyển của UAV (gồm lệnh vs thực tế). | ⭐ **Rất cao** |
| **`mavros-nav_info-errors`** | 44 file | Sai số bám mục tiêu do máy tính tự lái tính toán. | ⭐ **Rất cao** |
| **`mavctrl-rpy`** | 40 file | Lệnh điều khiển góc Roll-Pitch-Yaw mức thấp gửi đến mạch Pixhawk. | Khuyên dùng |
| **`mavctrl-path_dev`** | 38 file | Độ lệch khoảng cách so với đường bay thiết kế sẵn. | Khuyên dùng |

---

### 🔵 Nhóm 3: Định Vị Toàn Cầu & Vị Trí Nội Bộ (Localization & Waypoints)
*Nhóm này theo dõi tọa độ địa lý và hướng bay của UAV.*

| Tên File | Số lượng | Giải thích chức năng | Mức độ ưu tiên |
| :--- | :---: | :--- | :--- |
| **`mavros-local_position-pose`** | 40 file | Vị trí (X, Y, Z) và hướng xoay nội bộ dạng Quaternion. | Nên dùng |
| **`mavros-local_position-odom`** | 40 file | Dữ liệu Odometry tích hợp vị trí và vận tốc. | Nên dùng |
| **`mavros-local_position-velocity`** | 28 file | Vận tốc thực tế theo 3 trục mét/giây. | Nên dùng |
| **`mavros-global_position-global`** | 28 file | Tọa độ GPS thực tế (Vĩ độ, Kinh độ, Độ cao). | Cân nhắc |
| **`mavros-global_position-local`** | 28 file | Vị trí GPS quy đổi về hệ trục tọa độ mét. | Cân nhắc |
| **`mavros-global_position-rel_alt`** | 28 file | Độ cao tương đối so với điểm cất cánh. | Cân nhắc |
| **`mavros-global_position-compass_hdg`** | 28 file | Hướng mũi máy bay theo la bàn từ trường. | Cân nhắc |
| **`mavros-setpoint_raw-local`** | 34 file | Tọa độ điểm đích (Waypoints) nội bộ tiếp theo. | Ít dùng |
| **`mavros-setpoint_raw-target_global`** | 28 file | Tọa độ GPS của điểm đích tiếp theo. | Ít dùng |
| **`mavros-global_position-raw-fix`** | 28 file | Trạng thái bắt sóng và sai số của GPS thô. | Ít dùng |
| **`mavros-global_position-raw-gps_vel`** | 28 file | Vận tốc đo trực tiếp bằng cảm biến GPS. | Ít dùng |
| **`mavros-mission-reached`** | 3 file | Chỉ báo tín hiệu khi UAV bay qua một điểm đích. | Ít dùng |

---

### 🟢 Nhóm 4: Cảm Biến Vật Lý & Môi Trường (Sensors & Environment)
*Ghi nhận các lực tác động cơ học như gia tốc, từ trường, áp suất và gió.*

| Tên File | Số lượng | Giải thích chức năng | Mức độ ưu tiên |
| :--- | :---: | :--- | :--- |
| **`mavros-imu-data`** | 44 file | Gia tốc tuyến tính và vận tốc góc đã được lọc nhiễu. | Nên dùng |
| **`mavros-wind_estimation`** | 44 file | Ước lượng hướng gió và tốc độ gió thực tế ngoài trời. | Nên dùng |
| **`mavros-imu-data_raw`** | 44 file | Dữ liệu gia tốc và góc quay thô chưa qua bộ lọc. | Ít dùng |
| **`mavros-imu-mag`** | 44 file | Dữ liệu cảm biến từ trường (La bàn số). | Ít dùng |
| **`mavros-imu-atm_pressure`** | 44 file | Áp suất khí quyển (dùng để đo độ cao khí áp). | Ít dùng |
| **`mavros-imu-temperature`** | 44 file | Nhiệt độ của chip cảm biến IMU. | Có thể bỏ qua |

---

### 🟣 Nhóm 5: Hệ Thống Điện, Kênh Lái & Trạng Thái Hệ Thống
*Ghi nhận dòng điện, chế độ bay và lệnh điều khiển phần cứng của servo/động cơ.*

| Tên File | Số lượng | Giải thích chức năng | Mức độ ưu tiên |
| :--- | :---: | :--- | :--- |
| **`mavros-battery`** | 44 file | Điện áp, dòng điện tiêu thụ từ pin. | **Khuyên dùng** (Dòng sụt khi tắt động cơ) |
| **`mavros-vfr_hud`** | 44 file | Dữ liệu HUD (gồm tốc độ bay, ga động cơ dạng %). | **Khuyên dùng** |
| **`mavros-rc-out`** | 45 file | Tín hiệu PWM cấp ra các cánh lái và động cơ. | Nên dùng |
| **`mavros-state`** | 45 file | Chế độ bay (Auto/Manual) và trạng thái động cơ (Arm/Disarm). | Nên dùng |
| **`diagnostics`** | 46 file | Thông tin chẩn đoán phần cứng và hệ thống tự lái. | Ít dùng |
| **`mavros-rc-in`** | 45 file | Tín hiệu từ tay cầm điều khiển của phi công an toàn. | Ít dùng |
| **`mavros-time_reference`** | 44 file | Đồng bộ thời gian thực giữa máy tính nhúng và GPS. | Có thể bỏ qua |
| **`mavlink-from`** | 45 file | Gói tin MAVLink thô (rất nặng và phức tạp). | **Bỏ qua** |